<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/19_neural_networks/19_0_PyTorch_Basics/19_0_2_Autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Basics: Part 2
## Autograd — How PyTorch Computes Derivatives

---

## What This Notebook Is About

At the end of the last notebook, you saw a tensor with `requires_grad=True` build a silent record of every operation applied to it. The record was there — `y.grad_fn = <AddBackward0>` — but we did not explain what to do with it.

This notebook answers that question.

**The core problem:** To train a neural network with gradient descent, you need to know the derivative of the loss with respect to every single weight. In Unit 17, you visualized gradient descent as a ball rolling down a U-shaped bowl — the ball follows the slope, and the slope is the derivative. For a single linear regression model there was one slope, and it was simple enough to skip over the derivation. A neural network with 10,000 weights needs 10,000 slopes computed on every training step. A modern language model needs billions. Nobody computes these by hand.

PyTorch's answer is **autograd**: automatic differentiation. When you call `.backward()` on the loss, PyTorch traces the computation backwards through the graph it has been quietly building, applies the chain rule at each step, and deposits the result — one gradient per parameter — in each parameter's `.grad` attribute. The optimizer reads those gradients and updates the weights.

**What you will learn:**
1. How `.backward()` computes a derivative and stores it in `.grad`
2. How autograd handles multi-step computations using the chain rule
3. Why gradients accumulate by default — and the one-line fix
4. When to turn gradient tracking off with `torch.no_grad()`
5. How all of this maps to the five lines of the training loop you will write next notebook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.manual_seed(0)
print(f'PyTorch {torch.__version__}')

---

## Section 1: Computing a Derivative Automatically

Let's start with the simplest possible case: one number, one operation, one result. We want the derivative of y = x² with respect to x. Calculus tells us the answer is 2x — at x = 3, the derivative is 6. Let's see autograd compute that.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2    # forward pass

print('Forward pass:')
print(f'  x           = {x.item():.1f}')
print(f'  y = x²      = {y.item():.1f}')
print(f'  y.grad_fn   = {y.grad_fn}')   # records the operation that created y
print()

y.backward()   # backward pass: apply the chain rule from y back to every leaf

print('After y.backward():')
print(f'  x.grad      = {x.grad.item():.1f}')
print(f'  Calculus:   d(x²)/dx = 2x = 2×{x.item():.0f} = {2*x.item():.1f}  ✓')

The derivative of y = x² is dy/dx = 2x. At x = 3 that is 6.0 — exactly what `x.grad` holds.

Two things happened in that single `.backward()` call:
1. PyTorch read the `grad_fn` chain starting from `y`, working backwards toward `x`
2. At each step it applied the chain rule and accumulated the result into `x.grad`

The diagram below shows the structure of what PyTorch built during the forward pass and what it traversed during the backward pass. **Leaf nodes** — tensors you created directly with `requires_grad=True` — are where the final gradients land. The intermediate nodes (like `y`) store `grad_fn` but do not keep their own gradient by default; they are waypoints, not destinations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

for ax in axes:
    ax.set_xlim(0, 4)
    ax.set_ylim(0.2, 4.2)
    ax.axis('off')

def node(ax, x, y, text, color):
    ax.text(x, y, text, ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.4', fc=color, ec='#444', lw=1.5))

def arr(ax, x, y1, y2, color='#333', label=''):
    ax.annotate('', xy=(x, y2), xytext=(x, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0))
    if label:
        ax.text(x + 0.12, (y1 + y2) / 2, label, fontsize=9, color=color, va='center')

# --- Left: forward pass ---
axes[0].set_title('Step 1 — Forward Pass', fontweight='bold', fontsize=11)
node(axes[0], 2, 3.5, 'x = 3.0\n(leaf, requires_grad=True)', '#A9DFBF')
arr(axes[0],  2, 3.08, 2.42, label='x ** 2')
node(axes[0], 2, 2.0, 'y = 9.0\n(grad_fn = PowBackward0)', '#AED6F1')
axes[0].text(2, 0.85, 'Every operation is recorded\nin grad_fn as it runs.',
             ha='center', fontsize=9, color='#555', style='italic')

# --- Right: backward pass ---
axes[1].set_title('Step 2 — Backward Pass  (y.backward())', fontweight='bold', fontsize=11)
node(axes[1], 2, 3.5, 'x.grad = 6.0\nd(x²)/dx = 2×3 = 6', '#F9E79F')
arr(axes[1],  2, 2.42, 3.08, color='#C0392B', label='chain rule')
node(axes[1], 2, 2.0, 'y = 9.0\n(grad_fn = PowBackward0)', '#AED6F1')
axes[1].text(2, 0.85, 'Gradients flow ← from result\nback to every leaf node.',
             ha='center', fontsize=9, color='#C0392B', style='italic')

plt.suptitle('Autograd: the forward pass records, the backward pass differentiates',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

---

## Section 2: The Chain Rule Made Automatic

### Why We Need It

The y = x² example had a single operation. Real models chain many operations together — weights multiply, biases add, activation functions transform, and so on. To find the derivative of the final loss with respect to the very first weight, you have to trace through all of those steps.

The **chain rule** handles this. In plain language: if a tiny change in x causes a certain change in some intermediate value u, and a tiny change in u causes a certain change in z, then the total effect of a tiny change in x on z is the product of those two effects. Chain as many steps as you like — the same multiplication applies at each one.

Autograd applies this rule automatically, working backwards through the computation graph. Let's verify it on a concrete example by computing the answer by hand first.

### Hand Calculation: z = (ax + b)²

Let a = 2.0, b = 1.0, x = 3.0. We want dz/da and dz/db.

**Step 1 — name the intermediate value:**
Let u = ax + b = 2(3) + 1 = 7. Then z = u² = 49.

**Step 2 — derivative of the outer function (z = u²):**
dz/du = 2u = 2(7) = 14

**Step 3 — partial derivatives of the inner function (u = ax + b):**
du/da = x = 3
du/db = 1

**Step 4 — chain rule:**
dz/da = (dz/du)(du/da) = 14 × 3 = **42**
dz/db = (dz/du)(du/db) = 14 × 1 = **14**

We will ask PyTorch to verify.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)   # x is data, not a parameter — no requires_grad

z = (a * x + b) ** 2

print(f'z = (ax + b)² = ({a.item():.0f}×{x.item():.0f} + {b.item():.0f})² = {z.item():.0f}')
print()

z.backward()

print('Autograd vs. hand calculation:')
print(f'  dz/da = {a.grad.item():.1f}   (expected 42.0)')
print(f'  dz/db = {b.grad.item():.1f}   (expected 14.0)')

### Scaling Up: A Single Neuron

The formula `w1*x1 + w2*x2 + bias` is exactly how a single artificial neuron computes its output. Let's run the same autograd machinery on it and see what we get.

Notice what happens here: we compute the loss for one sample, call `.backward()` once, and immediately have the gradient of the loss with respect to every parameter — `w1`, `w2`, and `bias` — all in a single pass. That is the key capability that makes neural network training feasible.

In [ ]:
# One neuron: prediction = w1*x1 + w2*x2 + bias
# This is the building block of every layer in a neural network.
w1   = torch.tensor(2.0,  requires_grad=True)
w2   = torch.tensor(0.5,  requires_grad=True)
bias = torch.tensor(0.3,  requires_grad=True)

x1     = torch.tensor(3.0)
x2     = torch.tensor(2.0)
target = torch.tensor(1.0)

# Forward pass
prediction = w1*x1 + w2*x2 + bias
loss = (prediction - target) ** 2   # squared error

print(f'prediction = {prediction.item():.2f}')
print(f'target     = {target.item():.1f}')
print(f'loss       = (pred - target)² = {loss.item():.2f}')
print()

# Backward pass
loss.backward()

print('Gradients (how much each weight contributed to this error):')
print(f'  dloss/dw1   = {w1.grad.item():.4f}')
print(f'  dloss/dw2   = {w2.grad.item():.4f}')
print(f'  dloss/dbias = {bias.grad.item():.4f}')
print()
print('All three gradients are positive — the prediction is too high.')
print('Gradient descent subtracts a fraction of each gradient from each weight,')
print('pulling the prediction back toward the target.')

---

## Section 3: The Gradient Accumulation Trap

There is a behavior that catches almost every new PyTorch user at least once: **gradients accumulate by default**.

If you call `.backward()` a second time without clearing `.grad` first, PyTorch adds the new gradient on top of whatever was already there. This is intentional — there are legitimate uses, like accumulating gradients over many mini-batches when memory is tight. But in a normal training loop, it is almost always a bug, and it is silent. The loss goes down at first, then starts behaving strangely, and the mistake is hard to find.

Let's see it in action.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

print('Three steps without zeroing gradients first:')
for step in range(1, 4):
    y = x ** 2
    y.backward()
    print(f'  step {step}: x.grad = {x.grad.item():.1f}')

print()
print('We expect 6.0 at every step (dy/dx = 2x = 6, always).')
print('We got 6.0, 12.0, 18.0 — the gradient is stacking up.')

The fix is to zero the gradient at the start of each step. In a real training loop, the optimizer handles this for you with a single call.

The order is non-negotiable:
1. Zero the gradients from last step
2. Run the forward pass
3. Compute the loss
4. Call `.backward()`
5. Let the optimizer update the weights

Forgetting step 1 is the single most common silent bug in a first PyTorch training loop.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

print('Three steps with gradient zeroing:')
for step in range(1, 4):
    if x.grad is not None:
        x.grad.zero_()    # reset to zero before the next backward pass
    y = x ** 2
    y.backward()
    print(f'  step {step}: x.grad = {x.grad.item():.1f}')

print()
print('Now correct every step.')
print()
print('In a training loop, replace x.grad.zero_() with:')
print()
print('  optimizer.zero_grad()   # zeros .grad on every tracked parameter at once')
print('  loss.backward()         # computes fresh gradients')
print('  optimizer.step()        # updates every weight: w = w - lr * w.grad')

---

## Section 4: Turning Off Gradient Tracking

Building the computation graph takes memory and time. During a training step that cost is necessary — you need gradients to update the weights. But there are two places in a typical workflow where you run the model without updating anything:

1. **The validation loop** — you evaluate on held-out data after each epoch to check for overfitting. No gradients needed.
2. **Inference** — when the trained model makes predictions on new data.

`torch.no_grad()` is a context manager that suspends gradient tracking for everything inside it. Tensors computed inside the block have `grad_fn = None` and cannot call `.backward()`. In exchange, they use less memory and compute slightly faster. On a CPU the difference is modest; on a GPU with large batches it is meaningful.

The rule of thumb: wrap every forward pass that does not need a `.backward()` call in `torch.no_grad()`.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Training computation — gradient tracking is on
y_train = x ** 2 + 3 * x
print('Inside a normal computation (training):')
print(f'  y.grad_fn : {y_train.grad_fn}')   # AddBackward0 — tracked
print()

# Evaluation computation — gradient tracking is off
with torch.no_grad():
    y_eval = x ** 2 + 3 * x
print('Inside torch.no_grad() (validation / inference):')
print(f'  y.grad_fn : {y_eval.grad_fn}')    # None — not tracked
print()
print('Both tensors hold the same values:')
print(f'  y_train : {y_train.detach()}')
print(f'  y_eval  : {y_eval}')
print()
print('Use torch.no_grad() in: the validation loop, metric computation, inference.')

---

## Section 5: What This Means for a Neural Network

You now have all the pieces. Let's connect them to the training loop you will write in notebook 19_1_2.

A neural network is a composition of many operations — matrix multiplications, bias additions, activation functions — stacked in layers. Training it means solving the same problem you just solved on one neuron: find the gradient of the loss with respect to every weight, then subtract a fraction of that gradient from each weight. The only difference is scale.

A two-layer network with 100 hidden neurons has roughly 10,000 weights. A large network has millions. But the mechanism is identical to what you did above:

1. The **forward pass** builds the computation graph, just as `y = x**2` built it above.
2. Calling `loss.backward()` traverses the entire graph backward, applying the chain rule at every operation.
3. Every weight's `.grad` is populated — one number per weight, computed simultaneously.
4. The **optimizer** reads those gradients and updates each weight.

The elegance of autograd is that none of this changes as the network grows. A fifty-layer network calls `loss.backward()` exactly as you did above — just with a much deeper computation graph.

In [ ]:
# The five-line training loop you will write in notebook 19_1_2.
# Every line maps to something from this notebook.

# --- Training step ---
# optimizer.zero_grad()                      <- Section 3: clear last step's gradients
#
# predictions = model(X_batch)               <- forward pass builds the computation graph
# loss = loss_function(predictions, y_batch) <- one scalar value at the tip of the graph
#
# loss.backward()                            <- Section 1-2: autograd computes all gradients
#
# optimizer.step()                           <- reads every .grad, updates every weight

# --- Validation step (after each training epoch) ---
# with torch.no_grad():                      <- Section 4: no graph, no memory cost
#     val_preds = model(X_val)
#     val_loss  = loss_function(val_preds, y_val)

print('The five lines of the training loop are just autograd applied at scale.')
print('All five concepts appeared in this notebook.')

---

## Putting It All Together

| Concept | What it does | When you use it |
|---|---|---|
| `requires_grad=True` | Tells PyTorch to record operations on this tensor | On every model parameter |
| `y.grad_fn` | Records the operation that produced `y` | Built automatically during the forward pass |
| `y.backward()` | Traverses the computation graph; deposits gradients in each leaf's `.grad` | Once per training step, on the loss |
| `x.grad` | Holds the gradient of the loss with respect to `x` | Read by the optimizer to update `x` |
| Gradient accumulation | `.grad` adds up across `.backward()` calls by default | The reason you must zero gradients each step |
| `optimizer.zero_grad()` | Zeros `.grad` on every parameter before the next backward | First call in every training step |
| `torch.no_grad()` | Suspends the computation graph; saves memory and time | Validation loop and inference |

**Why this scales:** Autograd does not compute derivatives symbolically (like a textbook). It builds a graph of operations during the forward pass and applies the chain rule numerically — working backwards from the loss to each leaf. This is called **reverse-mode automatic differentiation**, or backpropagation. It computes the gradient with respect to every parameter in one backward pass, regardless of how many parameters there are. That is why `loss.backward()` on a network with a million weights takes roughly as long as a single forward pass — and why neural network training is computationally feasible at all.

**What is coming next:** In the next notebook, you will implement gradient descent from scratch on the penguin flipper-to-body-mass relationship from Unit 17. You will write the training loop, watch the loss curve converge, and discover firsthand how sensitive the algorithm is to the choice of learning rate.